# Station Stacking v14 - KDAL

Experimental notebook for `KDAL`.

This version keeps the v11 remaining-warmup target and Huber/ridge stack, computes v13 weather aggregates, trains only on a curated v11-plus-weather allowlist, and writes artifacts to `data/calibration/station_stacking_v14`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v14_curated_weather_stack"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V14_DROPPED_FEATURE_COLUMNS,
    V14_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V14 Contract

`feature_version="v14"` keeps the v11 remaining-warmup, Huber, and ridge-stack lineage, then adds only curated aggregate weather features that pass coverage.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V14_FEATURE_COLUMNS, sorted(V14_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,1992,2021-01-01,2026-06-21
7,KDAL,hrrr,1998,2021-01-01,2026-06-21
8,KDAL,nbm,1997,2021-01-01,2026-06-21


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v14",
    target_mode="remaining_warmup",
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v14",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v14/KDAL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:2544: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:2543: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:2544: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,730,1.445834,2.062259
1,validation_2024_2025,lightgbm,730,1.414785,2.039761
2,validation_2024_2025,catboost,730,1.490773,2.110274
3,validation_2024_2025,provider_mean,730,3.178604,3.977779
4,validation_2024_2025,provider_median,730,2.827166,3.671639
5,validation_2024_2025,nbm_raw,730,2.619924,3.489639
6,validation_2024_2025,hrrr_raw,730,5.584943,6.432032
7,validation_2024_2025,gfs_raw,730,2.717947,3.702726
8,test_2026,xgboost,172,1.555065,2.044428
9,test_2026,lightgbm,172,1.516831,1.973563


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/experiments/station_stacking_v14",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v14/model_weights/KDAL_station_high_regressor_v14_curated_weather_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v14/model_weights/KDAL_station_high_regressor_v14_curated_weather_stack.json'))

## Rain-Day V11 vs V14 MAE


In [9]:
def _rain_day_flags(features: pd.DataFrame) -> pd.DataFrame:
    work = features.copy()
    work["contract_date"] = pd.to_datetime(work["contract_date"]).dt.strftime("%Y-%m-%d")
    rain_signal = pd.Series(False, index=work.index)
    candidate_cols = [
        "observed_is_raining_at_as_of",
        "v4_observed_precip_any",
        "v4_any_forecast_precip",
        "gfs_forecast_has_precip",
        "hrrr_forecast_has_precip",
        "nbm_forecast_has_precip",
    ]
    for column in candidate_cols:
        if column in work:
            rain_signal |= pd.to_numeric(work[column], errors="coerce").fillna(0).gt(0)
    amount_cols = [
        column
        for column in work.columns
        if column.endswith("forecast_precip_total_mm")
        or column.endswith("forecast_precip_max_1h_mm")
        or column in {"observed_precip_recent_at_as_of", "precip_amount"}
    ]
    for column in amount_cols:
        rain_signal |= pd.to_numeric(work[column], errors="coerce").fillna(0).gt(0.01)
    return work.loc[rain_signal, ["contract_date"]].drop_duplicates()


def _prediction_mae_by_method(predictions: pd.DataFrame, version: str, rainy_dates: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame(columns=["version", "method", "rain_day_count", "mae_f", "rmse_f"])
    pred = predictions.copy()
    pred["contract_date"] = pd.to_datetime(pred["contract_date"]).dt.strftime("%Y-%m-%d")
    pred = pred.merge(rainy_dates, on="contract_date", how="inner")
    pred = pred.dropna(subset=["actual_high_f", "predicted_high_f"])
    if pred.empty:
        return pd.DataFrame(columns=["version", "method", "rain_day_count", "mae_f", "rmse_f"])
    pred["error_f"] = pred["actual_high_f"] - pred["predicted_high_f"]
    return (
        pred.groupby("method", dropna=False)
        .agg(
            rain_day_count=("contract_date", "nunique"),
            prediction_count=("contract_date", "size"),
            mae_f=("error_f", lambda s: float(np.mean(np.abs(s)))),
            rmse_f=("error_f", lambda s: float(np.sqrt(np.mean(np.square(s))))),
        )
        .reset_index()
        .assign(version=version)
        [["version", "method", "rain_day_count", "prediction_count", "mae_f", "rmse_f"]]
    )


v11_pred_path = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11" / f"{STATION_ID}_year_split_test_predictions.csv"
v11_feature_path = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11" / f"{STATION_ID}_features.csv"
v11_predictions = pd.read_csv(v11_pred_path) if v11_pred_path.exists() else pd.DataFrame()
rain_feature_frame = result.features if not result.features.empty else pd.read_csv(v11_feature_path)
rainy_dates = _rain_day_flags(rain_feature_frame)

rain_mae_comparison = pd.concat(
    [
        _prediction_mae_by_method(v11_predictions, "v11", rainy_dates),
        _prediction_mae_by_method(result.test_predictions, "v14", rainy_dates),
    ],
    ignore_index=True,
)
rain_mae_comparison = rain_mae_comparison.sort_values(["method", "version"]).reset_index(drop=True)
rain_mae_comparison


,version,method,rain_day_count,prediction_count,mae_f,rmse_f
0,v11,catboost,12,12,1.193048,1.868560
1,v14,catboost,21,21,1.722475,2.128076
2,v11,gfs_raw,12,12,4.019193,4.890454
3,v14,gfs_raw,21,21,3.178919,4.047596
4,v11,hrrr_raw,12,12,5.609713,7.296691
5,v14,hrrr_raw,21,21,4.973083,6.298146
6,v11,lightgbm,12,12,1.447380,2.186914
7,v14,lightgbm,21,21,1.483926,2.103726
8,v11,nbm_raw,12,12,3.859505,5.191775
9,v14,nbm_raw,21,21,3.229712,4.408020


In [10]:
rain_mae_wide = rain_mae_comparison.pivot_table(
    index="method",
    columns="version",
    values="mae_f",
    aggfunc="first",
)
if {"v11", "v14"}.issubset(rain_mae_wide.columns):
    rain_mae_wide["v14_minus_v11_mae_f"] = rain_mae_wide["v14"] - rain_mae_wide["v11"]
rain_mae_wide.sort_values("v14_minus_v11_mae_f" if "v14_minus_v11_mae_f" in rain_mae_wide else rain_mae_wide.columns[0])


version,v11,v14,v14_minus_v11_mae_f
method,,,
gfs_raw,4.019193,3.178919,-0.840274
hrrr_raw,5.609713,4.973083,-0.636630
nbm_raw,3.859505,3.229712,-0.629793
lightgbm,1.447380,1.483926,0.036547
ridge_stack,1.343985,1.477251,0.133266
xgboost,1.293249,1.677847,0.384598
catboost,1.193048,1.722475,0.529427
provider_mean,NaN,3.284252,NaN
provider_median,NaN,3.369709,NaN


## V14 Feature Coverage


In [11]:
v14_feature_coverage = (
    result.features[V14_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v14_feature_coverage


,feature,coverage_pct
0,v2_morning_warmup_to_consensus_f,100.000000
1,v2_humidity_warmup_interaction,100.000000
2,v2_spread_per_warmup_f,100.000000
3,observed_high_so_far_change_since_9am_f,100.000000
4,observed_morning_warmup_rate_f_per_hour,100.000000
...,...,...
62,v8_cloud_cover_mean_remaining_warmup_interaction,65.115115
63,v13_cloud_cover_remaining_warmup_interaction,65.115115
64,v13_precip_cloud_remaining_warmup_interaction,65.065065
65,v8_wind_gust_max_remaining_warmup_interaction,36.186186


In [12]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V14_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
163,v2_recent_heat_anomaly_f,numeric
...,...,...
217,climatology_high_10y_std_f,numeric
218,climatology_high_10y_count,numeric
219,provider_mean_minus_climatology_10y_f,numeric
220,observed_temp_minus_climatology_10y_f,numeric


## Dropped Feature Check


In [13]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V14_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [14]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,100.0
1,observed_temp_change_last_3h_f,100.0
2,observed_morning_warmup_rate_f_per_hour,100.0
3,observed_high_so_far_change_since_9am_f,100.0


## Rounded Within 1F Accuracy


In [15]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
8,oof_2026,xgboost,172,101,58.720930
3,oof_2026,lightgbm,172,99,57.558140
7,oof_2026,ridge_stack,172,98,56.976744
0,oof_2026,catboost,172,88,51.162791
4,oof_2026,nbm_raw,172,76,44.186047
6,oof_2026,provider_median,172,54,31.395349
1,oof_2026,gfs_raw,172,53,30.813953
5,oof_2026,provider_mean,172,43,25.000000
2,oof_2026,hrrr_raw,172,15,8.720930
12,validation_2024_2025,lightgbm,730,477,65.342466


## Version Comparison


In [16]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
    ("v13", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v13"),
    ("v14", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v14"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,138,1.369446,1.890651,v9
1,test_2026,xgboost,138,1.402066,1.889472,v11
2,test_2026,xgboost,138,1.419940,1.895994,v9
3,test_2026,ridge_stack,138,1.429989,1.907881,v11
4,test_2026,lightgbm,138,1.454606,1.932482,v11
...,...,...,...,...,...,...
117,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v5
118,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v6
119,validation_2024_2025,hrrr_raw,623,5.564150,6.352240,v1
120,validation_2024_2025,hrrr_raw,730,5.584943,6.432032,v13


## 2026 OOF Weather Brackets


In [17]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,172,1.555065,2.044428,40.116279
1,lightgbm,172,1.516831,1.973563,43.604651
2,catboost,172,1.736502,2.157421,34.883721
3,ridge_stack,172,1.532986,1.997736,42.44186
4,provider_mean,172,3.194053,3.954456,19.186047
5,provider_median,172,2.892289,3.729196,19.767442
6,nbm_raw,172,2.353419,3.203275,30.813953
7,hrrr_raw,172,5.595144,6.373455,5.232558
8,gfs_raw,172,2.914622,3.791390,22.674419
